# Finding research opportunities

A researcher looking for the next project usually gathers recent papers, notes what they leave open, and picks the questions that match their own expertise. SciLogic makes the first two steps mechanical: every encoding already lists its open items, and every item keeps the id of the statement it comes from, so any idea derived from it stays traceable to the paper.

Open items come from four places in an encoding, with two different origins:

| Item | Where in the encoding | Stated by |
|---|---|---|
| Limitation | `LIMITATION` statement | the authors |
| Outlook | `OUTLOOK` statement | the authors |
| Open question of an assumption | `open_question` of an `ASSUMPTION` | the annotator |
| Open proof obligation | `proof_obligations` with status `OPEN` | the annotator |

We use the ten arXiv preprints in `test_papers/`, taken from the cond-mat.mtrl-sci new submissions of 21 October 2026 and encoded by an LLM (Claude) from the PDFs of their first versions.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import yaml

sys.path.insert(0, "../scripts")
import render

pd.set_option("display.max_colwidth", None)

encodings = {}
for path in sorted(Path("../test_papers/yaml").glob("*.yaml")):
    with open(path) as f:
        chain = yaml.safe_load(f)
    encodings[chain["publication"]["arxiv"]] = chain

pd.DataFrame([
    {"arXiv": arxiv, "title": c["publication"]["title"],
     "statements": len(c["statements"]), "logic score": render.logic_score(c)[0]}
    for arxiv, c in encodings.items()
]).style.hide(axis="index").format(precision=2)

## Collect the open items

One function walks an encoding and returns its open items. Each item gets an id made of the arXiv number and the statement id (plus the obligation number for proof obligations), so it can always be looked up again.

In [ ]:
def open_items(arxiv, chain):
    items = []
    for s in chain["statements"]:
        sid = render.short(s["id"])
        kind = s["statement_type"]
        if kind in {"LIMITATION", "OUTLOOK"}:
            items.append((f"{arxiv}:{sid}", "authors", kind.lower(), s["text"]))
        if kind == "ASSUMPTION" and s.get("open_question"):
            items.append((f"{arxiv}:{sid}", "annotator", "assumption", s["open_question"]))
        for n, po in enumerate((s.get("assessment") or {}).get("proof_obligations", []), 1):
            if po["obligation_status"] == "OPEN":
                items.append((f"{arxiv}:{sid}/PO{n}", "annotator", "proof obligation", po["text"]))
    return items

items = pd.DataFrame(
    [item for arxiv, chain in encodings.items() for item in open_items(arxiv, chain)],
    columns=["item", "stated by", "kind", "text"],
).set_index("item")

print(f"{len(items)} open items from {len(encodings)} papers")
items.groupby(["stated by", "kind"]).size().rename("items").to_frame()

Without any deduplication, the ten encodings yield 62 open items: 29 stated by the authors (19 limitations and 10 outlook statements) and 33 formulated by the annotator (20 open questions of assumptions and 13 open proof obligations).

In [ ]:
items.reset_index().assign(paper=lambda d: d["item"].str.split(":").str[0]) \
    .pivot_table(index="paper", columns="kind", values="item", aggfunc="count", fill_value=0)

## Match them against a research interest

The next step needs judgment: which items can *this* researcher address? We gave the 62 items to Claude in a single pass, together with one research interest:

> computational materials science with atomistic simulation, reproducible workflows, and coupling to mesoscale models and experiment.

The call looks like this (it is not run when the book is built, so the book needs no API key):

```python
import anthropic
from claimchain.convert import DEFAULT_MODEL

interest = "computational materials science with atomistic simulation, reproducible workflows, ..."
prompt = (
    f"Research interest: {interest}\n\n"
    "For each open item below, answer ALONE (addressable with these methods alone), "
    "WITH_EXPERIMENT (only together with experiments) or OUTSIDE. Then propose studies "
    "that address several items at once, citing the item ids.\n\n"
    + items.to_csv()
)
reply = anthropic.Anthropic().messages.create(
    model=DEFAULT_MODEL, max_tokens=16000, messages=[{"role": "user", "content": prompt}],
)
```

In the LLM's judgment, 17 items can be addressed with the methods of this research interest alone, 18 only in combination with experiments, and 27 lie outside it. From these 35 items it derived five studies.

## Five derived studies

The table below gives the five studies and the open items each one addresses. The item ids are our mapping of the descriptions in the SciLogic paper (Table 5) onto the encodings, so every study can be traced back to the sentence in the paper that motivates it.

In [ ]:
studies = {
    "S1": ("Transferability of grain-boundary shear strengths across interatomic models",
           ["2610.28629:ASM1", "2610.28629:ASM2", "2610.28629:LIM1", "2610.28629:LIM2", "2610.28629:HYP1/PO2"]),
    "S2": ("Atomistically parameterized interfaces for phase-field models of Zr oxidation",
           ["2610.28493:ASM1", "2610.28493:LIM3", "2610.28493:ASM2", "2610.28493:ASM3", "2610.28493:HYP1/PO3"]),
    "S3": ("Size-dependent phase stability in nanocrystalline thin-film libraries",
           ["2610.28232:ASM3", "2610.28232:HYP1/PO3", "2610.27355:LIM1", "2610.27355:ASM2"]),
    "S4": ("Stability and storage mechanism of Co-doped Janus HfSSe",
           ["2610.28152:LIM1", "2610.28152:HYP1/PO4", "2610.28152:ASM1"]),
    "S5": ("Simulation-derived structural bases for differentiable powder diffraction",
           ["2610.28564:ASM2", "2610.28564:ASM1", "2610.28564:OUT1"]),
}

missing = [i for _, ids in studies.values() for i in ids if i not in items.index]
assert not missing, f"items not found in the encodings: {missing}"

pd.DataFrame([
    {"study": sid, "title": title, "items": len(ids),
     "source papers": ", ".join(sorted({i.split(":")[0] for i in ids}))}
    for sid, (title, ids) in studies.items()
]).style.hide(axis="index")

In [ ]:
addressed = sorted({i for _, ids in studies.values() for i in ids})
print(f"The five studies address {len(addressed)} of the {len(items)} open items.")

Each study, with the open items behind it and the paper they come from:

In [ ]:
for sid, (title, ids) in studies.items():
    papers = sorted({i.split(":")[0] for i in ids})
    print(f"{sid}  {title}")
    for p in papers:
        print(f"    from arXiv:{p}  {encodings[p]['publication']['title']}")
    for i in ids:
        row = items.loc[i]
        print(f"    - [{row['kind']}, {row['stated by']}] {row['text']}")
    print()

## A question no single paper asks

Most studies follow the open questions of one computational paper, as expected for a computational research interest. S3 is different. It combines two unrelated experimental papers: a combinatorial screening of fusion materials on co-sputtered films (2610.28232) and an on-chip annealing method for films with 3 to 7 nm grains (2610.27355). Both leave open whether nanocrystalline films retain phases that are not stable in bulk, but neither paper states this shared question. It only emerges when the open items of several papers are put side by side.

In [ ]:
items.loc[studies["S3"][1], ["kind", "text"]]

## Try it on your own papers

1. Encode the papers you want to scan: `claimchain paper.pdf -o encodings/paper.yaml` for each.
2. Point the loading cell above at your folder and rerun the notebook: `open_items` collects the open items, with ids that lead back to the paper.
3. Match the items against your own research interest, by hand or with an LLM call like the one above, and group them into studies.

**Caveats.** The encodings here are LLM drafts, the matching was a single LLM pass that did not re-read the papers, and the value of the proposed studies can only be judged by researchers in the field. What the encoding guarantees is traceability: every proposed study points to the exact statements that motivate it.